# Montar o app ExB "Mobilidade Urbana" (app 3) — v1
1. Padroniza os IDs das camadas do web map **Mobilidade Urbana** (IDs simples, sem acento — o ExB usa esses IDs).
2. Cria o web map **Mobilidade Urbana - Concentração de Pontos** (mapa de calor ligado) na pasta Mobilidade_Urbana.
3. Faz backup e escreve o `config.json` do app 3 no ExB Developer Edition: cabeçalho com menu + 4 páginas
   (Linhas e Horários · Concentração de Pontos · Sugestões · Sobre).
Depois: no construtor, **Ctrl+F5 → mudança pequena → Salvar → Publicar**.
Saída: `Documents\Saidas_Notebooks\Montar_ExB_Mobilidade_v1.txt`

In [ ]:
# ===== CÉLULA 1 — Configuração =====
import os, json, datetime, copy, shutil
NB_NOME = "Montar_ExB_Mobilidade_v1"
URL_APUC = "https://geo.apucarana.pr.gov.br/portal"
WEBMAP_ID = "c80cb05e18a645609f8f84740078b859"
WM_CALOR_TITULO = "Mobilidade Urbana - Concentração de Pontos"
PASTA_PORTAL = "Mobilidade_Urbana"
EXB_APP = r"C:\ExB_1.19\server\public\apps\3"
EXB_CFG = os.path.join(EXB_APP, "resources", "config", "config.json")
IDS_NOVOS = {"Bairros PMAPU": "bairros", "Eixos das Vias PMAPU": "eixos", "Linhas de ônibus": "linhas_onibus",
             "Concentração de pontos de ônibus": "calor_pontos", "Pontos de ônibus": "pontos_onibus",
             "Horários de ônibus": "horarios"}

home = os.path.expanduser("~")
_c = [os.path.join(home, "Documents", "Saidas_Notebooks"), os.path.join(home, "OneDrive", "Documents", "Saidas_Notebooks")]
PASTA_SAIDA = next((c for c in _c if os.path.isdir(c)), _c[0]); os.makedirs(PASTA_SAIDA, exist_ok=True)
PASTA_BKP = os.path.join(PASTA_SAIDA, f"backup_exb_mobilidade_{datetime.datetime.now():%Y%m%d_%H%M%S}")
os.makedirs(PASTA_BKP, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")
def log(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f: f.write(s + "\n")
def salvar(nome, obj):
    p = os.path.join(PASTA_BKP, nome)
    with open(p, "w", encoding="utf-8") as f: json.dump(obj, f, ensure_ascii=False, indent=2)
    return p

log("▶ CÉLULA 1 — Configuração: iniciando...")
if not os.path.isfile(EXB_CFG):
    raise RuntimeError("Não encontrei o config do app 3: " + EXB_CFG)
log("  App ExB:", EXB_APP)
log("  Backups:", PASTA_BKP)
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — Padronizar IDs das camadas do web map =====
log("\n▶ CÉLULA 2 — Padronização dos IDs do web map: iniciando...")
from arcgis.gis import GIS
gis = GIS("pro")
ME = gis.users.me
log("  Conectado como:", ME.username, "|", gis.url)
wm_item = gis.content.get(WEBMAP_ID)
wm = wm_item.get_data()
log("  Backup do web map:", salvar("webmap_mobilidade_antes.json", wm))
mud = 0
for grupo in ("operationalLayers", "tables"):
    for ol in wm.get(grupo, []) or []:
        novo = IDS_NOVOS.get(ol.get("title"))
        if novo and ol.get("id") != novo:
            log(f"  · {ol.get('title')}: id '{ol.get('id')}' → '{novo}'")
            ol["id"] = novo; mud += 1
ids = [ol.get("id") for ol in wm.get("operationalLayers", [])]
if "linhas_onibus" not in ids:
    raise RuntimeError("A camada 'Linhas de ônibus' não foi encontrada no web map.")
if mud:
    wm_item.update(item_properties={"text": json.dumps(wm)})
    log(f"  {mud} IDs atualizados.")
else:
    log("  IDs já estavam padronizados.")
log("  Camadas:", ids)
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — Web map de Concentração de Pontos =====
log("\n▶ CÉLULA 3 — Web map de concentração: iniciando...")
def _nome_pasta(f):
    return f.get("title") if isinstance(f, dict) else (getattr(f, "name", None) or getattr(f, "title", None))
def pasta_obj():
    for f in (ME.folders or []):
        if _nome_pasta(f) == PASTA_PORTAL:
            return f
    return None
def itens_na_pasta():
    try:
        return list(ME.items(folder=PASTA_PORTAL, max_items=500))
    except Exception:
        f = pasta_obj()
        return list(f.list()) if f is not None and hasattr(f, "list") else []
def adicionar(props):
    try:
        return gis.content.add(item_properties=props, folder=PASTA_PORTAL)
    except Exception:
        from arcgis.gis import ItemProperties, ItemTypeEnum
        ip = ItemProperties(title=props["title"], item_type=ItemTypeEnum.WEB_MAP, tags=props.get("tags"),
                            snippet=props.get("snippet"), text=props.get("text"))
        job = pasta_obj().add(item_properties=ip)
        return job.result() if hasattr(job, "result") else job
def compartilhar(item):
    try:
        item.sharing.sharing_level = "EVERYONE"
    except Exception:
        try: item.share(everyone=True)
        except Exception as e: log("  ! não consegui compartilhar como público:", e)

wm2 = copy.deepcopy(wm)
for ol in wm2.get("operationalLayers", []):
    if ol["id"] == "calor_pontos":
        ol["visibility"] = True
    elif ol["id"] == "pontos_onibus":
        ol["visibility"] = False
    elif ol["id"] == "linhas_onibus":
        ol["visibility"] = True; ol["opacity"] = 0.35
props = {"title": WM_CALOR_TITULO, "type": "Web Map", "text": json.dumps(wm2),
         "tags": "mobilidade, ônibus, pontos, mapa de calor, Apucarana, GeoAPUC",
         "snippet": "Concentração (mapa de calor) dos pontos de ônibus de Apucarana — página do app Mobilidade Urbana."}
exist = [i for i in itens_na_pasta() if i.type == "Web Map" and i.title == WM_CALOR_TITULO]
if exist:
    wm_calor = exist[0]
    wm_calor.update(item_properties={"text": props["text"]})
    log("  Já existia — conteúdo atualizado:", wm_calor.id)
else:
    wm_calor = adicionar(props)
    log("  Criado:", wm_calor.id)
compartilhar(wm_calor)
log("✔ CÉLULA 3 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 4 — Escrever o config do app 3 no ExB =====
log("\n▶ CÉLULA 4 — Escrita do config.json do app ExB: iniciando...")
CONFIG_TEMPLATE = r'''{
  "pages": {
    "page_0": {
      "id": "page_0",
      "mode": "FIT_WINDOW",
      "type": "NORMAL",
      "layout": {
        "LARGE": "layout_p0"
      },
      "isDefault": true,
      "isVisible": true,
      "label": "Linhas e Horários",
      "header": true
    },
    "page_calor": {
      "id": "page_calor",
      "mode": "FIT_WINDOW",
      "type": "NORMAL",
      "layout": {
        "LARGE": "layout_p_calor"
      },
      "isDefault": false,
      "isVisible": true,
      "label": "Concentração de Pontos",
      "header": true
    },
    "page_sugestoes": {
      "id": "page_sugestoes",
      "mode": "FIT_WINDOW",
      "type": "NORMAL",
      "layout": {
        "LARGE": "layout_p_sug"
      },
      "isDefault": false,
      "isVisible": true,
      "label": "Sugestões",
      "header": true
    },
    "page_sobre": {
      "id": "page_sobre",
      "mode": "FIT_WINDOW",
      "type": "NORMAL",
      "layout": {
        "LARGE": "layout_p_sobre"
      },
      "isDefault": false,
      "isVisible": true,
      "label": "Sobre",
      "header": true
    }
  },
  "layouts": {
    "layout_header": {
      "type": "FIXED",
      "content": {
        "0": {
          "type": "WIDGET",
          "widgetId": "widget_titulo",
          "isPending": false,
          "bbox": {
            "left": "1%",
            "top": "4px",
            "width": "30%",
            "height": "52px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "1": {
          "type": "WIDGET",
          "widgetId": "widget_nav1",
          "isPending": false,
          "bbox": {
            "left": "44%",
            "top": "10px",
            "width": "13%",
            "height": "40px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "2": {
          "type": "WIDGET",
          "widgetId": "widget_nav2",
          "isPending": false,
          "bbox": {
            "left": "58%",
            "top": "10px",
            "width": "15%",
            "height": "40px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "3": {
          "type": "WIDGET",
          "widgetId": "widget_nav3",
          "isPending": false,
          "bbox": {
            "left": "74%",
            "top": "10px",
            "width": "11%",
            "height": "40px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "4": {
          "type": "WIDGET",
          "widgetId": "widget_nav4",
          "isPending": false,
          "bbox": {
            "left": "86%",
            "top": "10px",
            "width": "9%",
            "height": "40px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        }
      },
      "order": [
        "0",
        "1",
        "2",
        "3",
        "4"
      ],
      "setting": {
        "lockDescendants": true
      }
    },
    "layout_card": {
      "type": "FIXED",
      "content": {
        "0": {
          "type": "WIDGET",
          "widgetId": "widget_card_cod",
          "isPending": false,
          "bbox": {
            "left": "3%",
            "top": "10px",
            "width": "18%",
            "height": "44px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false,
            "lockParent": true
          }
        },
        "1": {
          "type": "WIDGET",
          "widgetId": "widget_card_nome",
          "isPending": false,
          "bbox": {
            "left": "25%",
            "top": "4px",
            "width": "72%",
            "height": "28px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false,
            "lockParent": true
          }
        },
        "2": {
          "type": "WIDGET",
          "widgetId": "widget_card_sent",
          "isPending": false,
          "bbox": {
            "left": "25%",
            "top": "32px",
            "width": "72%",
            "height": "26px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false,
            "lockParent": true
          }
        }
      },
      "order": [
        "0",
        "1",
        "2"
      ],
      "label": "Padrão"
    },
    "layout_mapa1": {
      "type": "FIXED",
      "content": {},
      "order": []
    },
    "layout_p0": {
      "type": "FIXED",
      "content": {
        "0": {
          "type": "WIDGET",
          "widgetId": "widget_guia",
          "isPending": false,
          "bbox": {
            "left": "0.6%",
            "top": "1%",
            "width": "23%",
            "height": "9%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "1": {
          "type": "WIDGET",
          "widgetId": "widget_lista",
          "isPending": false,
          "bbox": {
            "left": "0.6%",
            "top": "11%",
            "width": "23%",
            "height": "45%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "2": {
          "type": "WIDGET",
          "widgetId": "widget_info",
          "isPending": false,
          "bbox": {
            "left": "0.6%",
            "top": "57%",
            "width": "23%",
            "height": "42%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "3": {
          "type": "WIDGET",
          "widgetId": "widget_mapa",
          "isPending": false,
          "bbox": {
            "left": "24.4%",
            "top": "1%",
            "width": "75%",
            "height": "98%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        }
      },
      "order": [
        "0",
        "1",
        "2",
        "3"
      ],
      "label": "Linhas e Horários"
    },
    "layout_mapa2": {
      "type": "FIXED",
      "content": {},
      "order": []
    },
    "layout_p_calor": {
      "type": "FIXED",
      "content": {
        "0": {
          "type": "WIDGET",
          "widgetId": "widget_mapa_calor",
          "isPending": false,
          "bbox": {
            "left": "0%",
            "top": "0%",
            "width": "100%",
            "height": "100%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "1": {
          "type": "WIDGET",
          "widgetId": "widget_calor_txt",
          "isPending": false,
          "bbox": {
            "left": "1.5%",
            "top": "3%",
            "width": "24%",
            "height": "19%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        }
      },
      "order": [
        "0",
        "1"
      ],
      "label": "Concentração de Pontos"
    },
    "layout_p_sug": {
      "type": "FIXED",
      "content": {
        "0": {
          "type": "WIDGET",
          "widgetId": "widget_sug_txt",
          "isPending": false,
          "bbox": {
            "left": "25%",
            "top": "15%",
            "width": "50%",
            "height": "45%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        }
      },
      "order": [
        "0"
      ],
      "label": "Sugestões"
    },
    "layout_p_sobre": {
      "type": "FIXED",
      "content": {
        "0": {
          "type": "WIDGET",
          "widgetId": "widget_sobre_txt",
          "isPending": false,
          "bbox": {
            "left": "25%",
            "top": "8%",
            "width": "50%",
            "height": "70%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        }
      },
      "order": [
        "0"
      ],
      "label": "Sobre"
    }
  },
  "exbVersion": "1.19.0",
  "mainSizeMode": "LARGE",
  "theme": "themes/default/",
  "customTheme": {
    "colors": {
      "primary": "#1A2942"
    },
    "focusedStyles": {
      "color": "#4285F4"
    }
  },
  "forBuilderAttributes": {
    "lockLayout": false
  },
  "template": "blankfullscreen",
  "widgets": {
    "widget_titulo": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Título do app",
      "id": "widget_titulo",
      "config": {
        "text": "<p style=\"line-height:1.2;margin:0\"><span style=\"font-size:20px;font-weight:700;color:#ffffff\">🚌 Mobilidade Urbana</span></p><p style=\"line-height:1.2;margin:0\"><span style=\"font-size:12px;color:#c9d3e6\">GeoAPUC · Transporte coletivo de Apucarana</span></p>",
        "style": {
          "verticalAlign": "center",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-2"
        }
      }
    },
    "widget_nav1": {
      "uri": "widgets/common/button/",
      "version": "1.19.0",
      "id": "widget_nav1",
      "label": "Menu – Linhas e Horários",
      "config": {
        "functionConfig": {
          "text": "Linhas e Horários",
          "toolTip": "Lista de linhas, mapa e horários",
          "linkParam": {
            "linkType": "PAGE",
            "value": "page_0",
            "openType": "_self"
          }
        },
        "styleConfig": {
          "useCustom": true,
          "customStyle": {
            "regular": {
              "background": {
                "color": "rgba(255,255,255,0.08)",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#ffffff",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "14px",
                "bold": true
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#ffffff",
                "size": 14
              }
            },
            "hover": {
              "background": {
                "color": "#F5A623",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#ffffff",
                "width": "0px"
              },
              "text": {
                "color": "#1A2942",
                "size": "14px",
                "bold": true
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#1A2942",
                "size": 14
              }
            }
          }
        }
      }
    },
    "widget_nav2": {
      "uri": "widgets/common/button/",
      "version": "1.19.0",
      "id": "widget_nav2",
      "label": "Menu – Concentração de Pontos",
      "config": {
        "functionConfig": {
          "text": "Concentração de Pontos",
          "toolTip": "Mapa de calor dos pontos de ônibus",
          "linkParam": {
            "linkType": "PAGE",
            "value": "page_calor",
            "openType": "_self"
          }
        },
        "styleConfig": {
          "useCustom": true,
          "customStyle": {
            "regular": {
              "background": {
                "color": "rgba(255,255,255,0.08)",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#ffffff",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "14px",
                "bold": true
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#ffffff",
                "size": 14
              }
            },
            "hover": {
              "background": {
                "color": "#F5A623",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#ffffff",
                "width": "0px"
              },
              "text": {
                "color": "#1A2942",
                "size": "14px",
                "bold": true
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#1A2942",
                "size": 14
              }
            }
          }
        }
      }
    },
    "widget_nav3": {
      "uri": "widgets/common/button/",
      "version": "1.19.0",
      "id": "widget_nav3",
      "label": "Menu – Sugestões",
      "config": {
        "functionConfig": {
          "text": "Sugestões",
          "toolTip": "Envie sugestões sobre o transporte coletivo",
          "linkParam": {
            "linkType": "PAGE",
            "value": "page_sugestoes",
            "openType": "_self"
          }
        },
        "styleConfig": {
          "useCustom": true,
          "customStyle": {
            "regular": {
              "background": {
                "color": "rgba(255,255,255,0.08)",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#ffffff",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "14px",
                "bold": true
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#ffffff",
                "size": 14
              }
            },
            "hover": {
              "background": {
                "color": "#F5A623",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#ffffff",
                "width": "0px"
              },
              "text": {
                "color": "#1A2942",
                "size": "14px",
                "bold": true
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#1A2942",
                "size": 14
              }
            }
          }
        }
      }
    },
    "widget_nav4": {
      "uri": "widgets/common/button/",
      "version": "1.19.0",
      "id": "widget_nav4",
      "label": "Menu – Sobre",
      "config": {
        "functionConfig": {
          "text": "Sobre",
          "toolTip": "Fonte dos dados e créditos",
          "linkParam": {
            "linkType": "PAGE",
            "value": "page_sobre",
            "openType": "_self"
          }
        },
        "styleConfig": {
          "useCustom": true,
          "customStyle": {
            "regular": {
              "background": {
                "color": "rgba(255,255,255,0.08)",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#ffffff",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "14px",
                "bold": true
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#ffffff",
                "size": 14
              }
            },
            "hover": {
              "background": {
                "color": "#F5A623",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#ffffff",
                "width": "0px"
              },
              "text": {
                "color": "#1A2942",
                "size": "14px",
                "bold": true
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#1A2942",
                "size": 14
              }
            }
          }
        }
      }
    },
    "widget_guia": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Guia da lista",
      "id": "widget_guia",
      "config": {
        "text": "<p style=\"margin:0\"><span style=\"font-size:15px;font-weight:700;color:#1A2942\">Linhas de ônibus</span></p><p style=\"margin:0\"><span style=\"font-size:12px;color:#5b6475\">Busque pelo número ou nome. Clique numa linha para ver o trajeto e os horários (dias úteis).</span></p>",
        "style": {
          "verticalAlign": "start",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-2"
        }
      },
      "style": {
        "background": {
          "color": "#f4f6fb",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_card_cod": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Cartão – nº da linha",
      "id": "widget_card_cod",
      "config": {
        "text": "<p style=\"text-align:center;line-height:1;margin:0\"><span style=\"font-size:20px;font-weight:700;color:#ffffff\"><exp data-uniqueid=\"aa7f7bee_6c68_4962_a121_54a3d9853a64\" data-dsid=\"dataSource_1-linhas_onibus\" data-expression=\"%7B%22name%22%3A%22%7BCOD_LINHA%7D%22%2C%22parts%22%3A%5B%7B%22type%22%3A%22FIELD%22%2C%22exp%22%3A%22%7BCOD_LINHA%7D%22%2C%22dataSourceId%22%3A%22dataSource_1-linhas_onibus%22%2C%22jimuFieldName%22%3A%22cod_linha%22%2C%22isFromRepeatedDataSourceContext%22%3Atrue%7D%5D%7D\">﻿<span contenteditable=\"false\">﻿﻿{COD_LINHA}﻿﻿</span>﻿</exp></span></p>",
        "style": {
          "verticalAlign": "center",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-2"
        }
      },
      "style": {
        "background": {
          "color": "#1A2942",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            8,
            8,
            8,
            8
          ],
          "unit": "px"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1-linhas_onibus",
          "mainDataSourceId": "dataSource_1-linhas_onibus",
          "rootDataSourceId": "dataSource_1",
          "fields": [
            "cod_linha"
          ]
        }
      ],
      "useDataSourcesEnabled": true
    },
    "widget_card_nome": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Cartão – nome",
      "id": "widget_card_nome",
      "config": {
        "text": "<p style=\"margin:0\"><span style=\"font-size:14px;font-weight:600;color:#1A2942\"><exp data-uniqueid=\"31851310_a3c9_4f85_ae1d_bf62e3237fff\" data-dsid=\"dataSource_1-linhas_onibus\" data-expression=\"%7B%22name%22%3A%22%7BNOME%7D%22%2C%22parts%22%3A%5B%7B%22type%22%3A%22FIELD%22%2C%22exp%22%3A%22%7BNOME%7D%22%2C%22dataSourceId%22%3A%22dataSource_1-linhas_onibus%22%2C%22jimuFieldName%22%3A%22nome%22%2C%22isFromRepeatedDataSourceContext%22%3Atrue%7D%5D%7D\">﻿<span contenteditable=\"false\">﻿﻿{NOME}﻿﻿</span>﻿</exp></span></p>",
        "style": {
          "verticalAlign": "start",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-2"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1-linhas_onibus",
          "mainDataSourceId": "dataSource_1-linhas_onibus",
          "rootDataSourceId": "dataSource_1",
          "fields": [
            "nome"
          ]
        }
      ],
      "useDataSourcesEnabled": true
    },
    "widget_card_sent": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Cartão – sentido",
      "id": "widget_card_sent",
      "config": {
        "text": "<p style=\"margin:0\"><span style=\"font-size:12px;color:#6b7690\"><exp data-uniqueid=\"f971231f_88ab_41db_aec3_65e0aa29363d\" data-dsid=\"dataSource_1-linhas_onibus\" data-expression=\"%7B%22name%22%3A%22%7BSENTIDO%7D%22%2C%22parts%22%3A%5B%7B%22type%22%3A%22FIELD%22%2C%22exp%22%3A%22%7BSENTIDO%7D%22%2C%22dataSourceId%22%3A%22dataSource_1-linhas_onibus%22%2C%22jimuFieldName%22%3A%22sentido%22%2C%22isFromRepeatedDataSourceContext%22%3Atrue%7D%5D%7D\">﻿<span contenteditable=\"false\">﻿﻿{SENTIDO}﻿﻿</span>﻿</exp></span></p>",
        "style": {
          "verticalAlign": "start",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-2"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1-linhas_onibus",
          "mainDataSourceId": "dataSource_1-linhas_onibus",
          "rootDataSourceId": "dataSource_1",
          "fields": [
            "sentido"
          ]
        }
      ],
      "useDataSourcesEnabled": true
    },
    "widget_lista": {
      "uri": "widgets/common/list/",
      "version": "1.19.0",
      "id": "widget_lista",
      "label": "Lista de linhas",
      "config": {
        "maxItems": 200,
        "space": 6,
        "verticalSpace": 6,
        "horizontalSpace": 10,
        "itemsPerPage": 20,
        "isItemStyleConfirm": true,
        "isInitialed": true,
        "pageStyle": "SCROLL",
        "itemStyle": "STYLE5",
        "hoverType": "HOVER0",
        "selectedStyle": "STYLE0",
        "differentOddEven": false,
        "scrollBarOpen": true,
        "direction": "VERTICAL",
        "alignType": "CENTER",
        "scrollStep": 2,
        "cardConfigs": {
          "DEFAULT": {
            "cardSize": {
              "LARGE": {
                "width": 420,
                "height": 64
              }
            },
            "enable": true,
            "backgroundStyle": {
              "border": {
                "border": {
                  "type": "solid",
                  "color": "",
                  "width": "0px"
                }
              },
              "borderRadius": {
                "number": [
                  8,
                  8,
                  8,
                  8
                ],
                "unit": "px"
              },
              "background": {
                "color": "#ffffff",
                "fillType": "fill",
                "image": {
                  "url": ""
                }
              }
            },
            "listLayout": "Custom"
          },
          "HOVER": {
            "cardSize": {
              "LARGE": {
                "width": 420,
                "height": 64
              }
            },
            "enable": true,
            "backgroundStyle": {
              "border": {
                "border": {
                  "type": "solid",
                  "color": "",
                  "width": "0px"
                }
              },
              "borderRadius": {
                "number": [
                  8,
                  8,
                  8,
                  8
                ],
                "unit": "px"
              },
              "background": {
                "color": "#eaf0fb",
                "fillType": "fill",
                "image": {
                  "url": ""
                }
              }
            },
            "listLayout": "Auto"
          },
          "SELECTED": {
            "cardSize": {
              "LARGE": {
                "width": 420,
                "height": 64
              }
            },
            "backgroundStyle": {
              "border": {
                "borderLeft": {
                  "type": "solid",
                  "color": "#F5A623",
                  "width": "5px"
                }
              },
              "borderRadius": {
                "number": [
                  8,
                  8,
                  8,
                  8
                ],
                "unit": "px"
              },
              "background": {
                "color": "rgba(66,133,244,0.12)",
                "fillType": "fill",
                "image": {
                  "url": ""
                }
              }
            },
            "selectionMode": "SINGLE",
            "listLayout": "Auto"
          }
        },
        "style": {
          "id": "default"
        },
        "isCheckEmptyTemplate": false,
        "filters": null,
        "isShowAutoRefresh": false,
        "layoutType": "ROW",
        "keepAspectRatio": false,
        "searchOpen": true,
        "sortOpen": false,
        "showRefresh": false,
        "showSelectedOnlyOpen": false,
        "showClearSelected": true,
        "showRecordCount": true,
        "filterOpen": false,
        "sqlExprObj": null,
        "sortFields": null,
        "searchFields": [
          "id_rota",
          "nome",
          "sentido"
        ],
        "searchHint": "Nº ou nome da linha",
        "hidePageTotal": false,
        "navigatorOpen": false,
        "lockItemRatio": true
      },
      "dataActions": {
        "setFilter": {
          "enabled": true
        },
        "relatedData": {
          "enabled": true
        }
      },
      "useDataSourcesEnabled": true,
      "layouts": {
        "DEFAULT": {
          "LARGE": "layout_card"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1-linhas_onibus",
          "mainDataSourceId": "dataSource_1-linhas_onibus",
          "rootDataSourceId": "dataSource_1",
          "fields": [
            "cod_linha",
            "nome",
            "sentido",
            "id_rota"
          ]
        }
      ],
      "outputDataSources": [],
      "style": {
        "background": {
          "color": "#ffffff",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_info": {
      "uri": "widgets/arcgis/feature-info/",
      "version": "1.19.0",
      "id": "widget_info",
      "label": "Horários da linha",
      "config": {
        "title": true,
        "fields": true,
        "media": true,
        "attachments": false,
        "lastEditInfo": false,
        "dsNavigator": false,
        "useMapWidget": false,
        "styleType": "usePopupDefined",
        "noDataMessage": "Selecione uma linha na lista para ver os horários."
      },
      "useDataSourcesEnabled": true,
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1-linhas_onibus",
          "mainDataSourceId": "dataSource_1-linhas_onibus",
          "rootDataSourceId": "dataSource_1"
        }
      ],
      "style": {
        "background": {
          "color": "#ffffff",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_mapa": {
      "uri": "widgets/arcgis/arcgis-map/",
      "version": "1.19.0",
      "id": "widget_mapa",
      "label": "Mapa – Linhas e Horários",
      "config": {
        "toolConfig": {
          "canZoom": true,
          "canHome": true,
          "canSearch": false,
          "canNavigation": true,
          "canLocate": true,
          "canCompass": false,
          "canFullScreen": true,
          "canScaleBar": true,
          "canLayers": true,
          "canBaseMap": true
        },
        "isUseCustomMapState": false,
        "initialMapDataSourceID": "dataSource_1",
        "initialMapState": null,
        "scaleRange": null
      },
      "layouts": {
        "MapFixedLayout": {
          "LARGE": "layout_mapa1"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1",
          "mainDataSourceId": "dataSource_1"
        }
      ]
    },
    "widget_mapa_calor": {
      "uri": "widgets/arcgis/arcgis-map/",
      "version": "1.19.0",
      "id": "widget_mapa_calor",
      "label": "Mapa – Concentração de pontos",
      "config": {
        "toolConfig": {
          "canZoom": true,
          "canHome": true,
          "canSearch": false,
          "canNavigation": true,
          "canLocate": true,
          "canCompass": false,
          "canFullScreen": true,
          "canScaleBar": true,
          "canLayers": true,
          "canBaseMap": true
        },
        "isUseCustomMapState": false,
        "initialMapDataSourceID": "dataSource_2",
        "initialMapState": null,
        "scaleRange": null
      },
      "layouts": {
        "MapFixedLayout": {
          "LARGE": "layout_mapa2"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_2",
          "mainDataSourceId": "dataSource_2"
        }
      ]
    },
    "widget_calor_txt": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Explicação do mapa de calor",
      "id": "widget_calor_txt",
      "config": {
        "text": "<p style=\"margin:0 0 6px 0\"><span style=\"font-size:16px;font-weight:700;color:#1A2942\">Concentração de pontos de ônibus</span></p><p style=\"margin:0 0 6px 0\"><span style=\"font-size:12px;color:#3d4657\">Quanto mais <b style=\"color:#d4a000\">amarelo</b>, mais pontos de ônibus próximos uns dos outros — são os corredores com maior oferta de transporte coletivo. Tons <b style=\"color:#7d5ba6\">lilás</b> indicam pontos mais espaçados.</span></p><p style=\"margin:0\"><span style=\"font-size:11px;color:#6b7690\">Fonte: setor de transportes · 1.860 pontos</span></p>",
        "style": {
          "verticalAlign": "start",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-2"
        }
      },
      "style": {
        "background": {
          "color": "#ffffff",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_sug_txt": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Sugestões – aviso (substituir pelo Survey123)",
      "id": "widget_sug_txt",
      "config": {
        "text": "<p style=\"text-align:center;margin:0 0 10px 0\"><span style=\"font-size:40px\">💬</span></p><p style=\"text-align:center;margin:0 0 8px 0\"><span style=\"font-size:22px;font-weight:700;color:#1A2942\">Sugestões sobre o transporte coletivo</span></p><p style=\"text-align:center;margin:0 0 14px 0\"><span style=\"font-size:14px;color:#3d4657\">Em breve você poderá enviar sugestões de novas linhas, pontos e horários diretamente por aqui, marcando o local no mapa.</span></p><p style=\"text-align:center;margin:0\"><span style=\"font-size:13px;font-weight:700;color:#ffffff;background:#F5A623;padding:4px 12px;border-radius:12px\">EM BREVE</span></p>",
        "style": {
          "verticalAlign": "center",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-2"
        }
      },
      "style": {
        "background": {
          "color": "#ffffff",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_sobre_txt": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Sobre – texto",
      "id": "widget_sobre_txt",
      "config": {
        "text": "<p style=\"margin:0 0 10px 0\"><span style=\"font-size:22px;font-weight:700;color:#1A2942\">Sobre este aplicativo</span></p><p style=\"margin:0 0 8px 0\"><span style=\"font-size:14px;color:#3d4657\">O <b>Mobilidade Urbana</b> reúne as linhas, os pontos e os horários do transporte coletivo de Apucarana num só lugar, para consulta rápida pelo cidadão.</span></p><p style=\"margin:0 0 4px 0\"><span style=\"font-size:14px;font-weight:700;color:#1A2942\">Fonte dos dados</span></p><p style=\"margin:0 0 8px 0\"><span style=\"font-size:13px;color:#3d4657\">Itinerários, pontos e quadro de horários: setor de transportes da Prefeitura de Apucarana. Bairros e eixos de vias: base cartográfica PMAPU.</span></p><p style=\"margin:0 0 4px 0\"><span style=\"font-size:14px;font-weight:700;color:#1A2942\">Antes de usar</span></p><p style=\"margin:0 0 8px 0\"><span style=\"font-size:13px;color:#3d4657\">Os horários são de <b>dias úteis</b>. Quando um sentido não tem horários próprios, são mostrados os horários de saída da mesma linha nos outros sentidos.</span></p><p style=\"margin:0\"><span style=\"font-size:12px;color:#6b7690\">Elaboração: GeoAPUC · Prefeitura de Apucarana</span></p>",
        "style": {
          "verticalAlign": "start",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-2"
        }
      },
      "style": {
        "background": {
          "color": "#ffffff",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    }
  },
  "widgetsManifest": {},
  "views": {},
  "sections": {},
  "dialogs": {},
  "messageConfigs": {
    "messageConfig_lista_zoom": {
      "id": "messageConfig_lista_zoom",
      "widgetId": "widget_lista",
      "messageType": "DATA_RECORDS_SELECTION_CHANGE",
      "actions": [
        {
          "actionId": "widget_mapa-zoomToFeature-1",
          "widgetId": "widget_mapa",
          "messageWidgetId": "widget_lista",
          "actionName": "zoomToFeature",
          "description": null,
          "version": "1.19.0",
          "config": {
            "useDataSource": {
              "dataSourceId": "dataSource_1-linhas_onibus",
              "mainDataSourceId": "dataSource_1-linhas_onibus",
              "rootDataSourceId": "dataSource_1"
            },
            "useDataSources": [
              {
                "dataSourceId": "dataSource_1-linhas_onibus",
                "mainDataSourceId": "dataSource_1-linhas_onibus",
                "rootDataSourceId": "dataSource_1"
              }
            ],
            "goToInitialMapExtentWhenSelectionCleared": true,
            "useAnyTriggerData": true
          }
        },
        {
          "actionId": "widget_mapa-flash-2",
          "widgetId": "widget_mapa",
          "messageWidgetId": "widget_lista",
          "actionName": "flash",
          "description": null,
          "version": "1.19.0",
          "config": {
            "useAnyTriggerData": true,
            "messageUseDataSource": {
              "dataSourceId": "dataSource_1-linhas_onibus",
              "mainDataSourceId": "dataSource_1-linhas_onibus",
              "rootDataSourceId": "dataSource_1"
            },
            "actionUseDataSource": null,
            "sqlExprObj": null,
            "enabledDataRelationShip": true,
            "connectionType": "SET_CUSTOM_FIELDS"
          }
        }
      ]
    }
  },
  "attributes": {
    "portalUrl": "https://geo.apucarana.pr.gov.br/portal"
  },
  "header": {
    "layout": {
      "LARGE": "layout_header"
    },
    "height": {
      "LARGE": "60px"
    },
    "backgroundColor": "#1A2942"
  },
  "sharedThemeVariables": {
    "header": {},
    "body": {},
    "button": {},
    "logo": {
      "small": "",
      "link": ""
    }
  },
  "mainLocale": "pt-br",
  "dataSources": {
    "dataSource_1": {
      "id": "dataSource_1",
      "type": "WEB_MAP",
      "sourceLabel": "Mobilidade Urbana",
      "itemId": "c80cb05e18a645609f8f84740078b859",
      "portalUrl": "https://geo.apucarana.pr.gov.br/portal",
      "isHidden": false,
      "childDataSourceJsons": {
        "linhas_onibus": {
          "id": "dataSource_1-linhas_onibus",
          "type": "FEATURE_LAYER",
          "sourceLabel": "Linhas de ônibus",
          "portalUrl": "https://geo.apucarana.pr.gov.br/portal",
          "itemId": "8c163a4936e143be96f69b58a2911ce8",
          "layerId": "0",
          "url": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Linhas_Onibus/FeatureServer/0"
        }
      }
    },
    "dataSource_2": {
      "id": "dataSource_2",
      "type": "WEB_MAP",
      "sourceLabel": "Mobilidade Urbana - Concentração de Pontos",
      "itemId": "__CALOR_ID__",
      "portalUrl": "https://geo.apucarana.pr.gov.br/portal",
      "isHidden": false
    }
  },
  "pageStructure": [
    {
      "page_0": []
    },
    {
      "page_calor": []
    },
    {
      "page_sugestoes": []
    },
    {
      "page_sobre": []
    }
  ]
}'''

cfg = json.loads(CONFIG_TEMPLATE.replace("__CALOR_ID__", wm_calor.id))
for nome in ("config.json", os.path.join("resources", "config", "config.json"), "info.json"):
    origem = os.path.join(EXB_APP, nome)
    if os.path.isfile(origem):
        destino = os.path.join(PASTA_BKP, "app3_" + nome.replace(os.sep, "_"))
        shutil.copy2(origem, destino)
        log("  Backup:", destino)
cfg["timestamp"] = int(datetime.datetime.now().timestamp() * 1000)
with open(EXB_CFG, "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)
log("  config.json escrito:", EXB_CFG)
log("  Páginas:", " | ".join(p["label"] for p in cfg["pages"].values()))
log("  Widgets:", len(cfg["widgets"]), "| Fontes de dados: web map", WEBMAP_ID, "+ web map", wm_calor.id)
log("✔ CÉLULA 4 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 5 — Próximos passos e links =====
log("\n▶ CÉLULA 5 — Links: iniciando...")
log("  PRÓXIMOS PASSOS no construtor:")
log("   1) Abra o construtor do app 3 e tecle Ctrl+F5")
log("   2) Faça uma mudança pequena (ex.: arraste 1px um widget e volte) → Salvar → Publicar")
log("   3) Teste: clique numa linha da lista → o mapa aproxima e o painel de baixo mostra os horários")
log("\n  ===== LINKS =====")
log("   Construtor (app 3):        https://localhost:3001/builder/?id=3")
log("   Prévia local:              https://localhost:3001/experience/3/?locale=pt-br")
log(f"   Web map principal:         {URL_APUC}/apps/mapviewer/index.html?webmap={WEBMAP_ID}")
log(f"   Web map concentração:      {URL_APUC}/apps/mapviewer/index.html?webmap={wm_calor.id}")
log(f"   Item web map concentração: {URL_APUC}/home/item.html?id={wm_calor.id}")
log(f"\n  Backups em: {PASTA_BKP}")
log(f"  Saída completa: {ARQ_TXT}")
log("✔ CÉLULA 5 concluída com sucesso.")
